# 2. Preprocessing

*AI assistance (Claude-sonnet-4.6 in UvA AI Chat): notebook structure and bot-role matching logic.*

This notebook takes the raw data collected in notebook 01 and runs four steps in order:

| Step | Input | Output |
|------|-------|--------|
| 1 | `sample_index.json`, `talkpage_messages.jsonl` | Cleaned newcomer dataframe |
| 2 | `newcomer_edits.jsonl` + MediaWiki API | Retention labels (RQ1 DV) |
| 3 | Zheng et al. (2019) bot classification file | Bot-role contact dummies (RQ1 IV) |
| 4 | ConvoKit politeness scores (Step 3 input) | Per-sender politeness profiles (RQ2 DV) |


<div style="padding: 15px; border-left: 5px solid #0366d6; background-color: #f6f8fa; border-radius: 4px; font-family: sans-serif; line-height: 1.5;">
    <strong style="color: #0366d6; font-size: 1.1em;">⚠️ Sampling Note: Temporal Constraint</strong>
    <p style="margin: 10px 0;">
        The 30,000-account cap was reached between <b>May 25–31, 2025</b> due to its high registration volume.
    </p>
    <ul style="margin: 0; padding-left: 20px;">
        <li><b>Original Plan:</b> March–May 2025</li>
        <li><b>Actual Cohort:</b> May 25–31, 2025</li>
        <li><b>Retention Check:</b> Concludes July 24–31, 2025</li>
    </ul>
</div>

### 2.0 Configuration

In [ ]:
import json
import os
import time
from pathlib import Path
from datetime import datetime, timedelta
import requests
import pandas as pd
import zipfile

In [ ]:
# ── Working directory ────────────────────────────────────────────────────────
os.chdir("/Users/ada/bdacaII-2026-ada")

# ── API endpoint ──────────────────────────────────────────────────────────────
API = "https://en.wikipedia.org/w/api.php"

# ── Retention window (must match notebook 1) ─────────────────────────────────
RETENTION_DAYS = 60   # days after first edit to check retention

# ── Polite API call settings (must match notebook 1) ─────────────────────────
MAXLAG              = 5    # seconds; cooperative DB-lag threshold
REQUEST_TIMEOUT     = 15   # seconds before a request is considered hung
RETRY_DELAY         = 2    # seconds to wait between retry attempts
MAX_RETRIES         = 5    # maximum retry attempts per API call
INTER_REQUEST_DELAY = 0.1  # polite pause between individual API calls (seconds)
INTER_PAGE_DELAY    = 0.5  # polite pause between paginated API calls (seconds)

# ── Bot matching settings ─────────────────────────────────────────────────────
# Bots not matched to Zheng et al. (2019) are cross-checked against the live
# Wikipedia bot category before being classified as unidentified.
WIKI_BOT_CATEGORY = "Category:All Wikipedia bots"

# ── Data directories (must match notebook 1) ──────────────────────────────────
DATA_DIR      = Path("Final_project/data")
RAW_DIR       = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

# ── Input paths ───────────────────────────────────────────────────────────────
SAMPLE_INDEX_PATH      = PROCESSED_DIR / "sample_index.json"
TALKPAGE_JSONL         = RAW_DIR / "talkpage_messages.jsonl"
BOT_CLASSIFICATION_PATH = RAW_DIR / "botrole_predict_and_proportion_of_edits.xlsx"

# ── Output paths ──────────────────────────────────────────────────────────────
NEWCOMERS_CSV    = PROCESSED_DIR / "newcomers.csv"
ANALYSIS_CSV     = PROCESSED_DIR / "analysis_dataset.csv"
MESSAGES_CSV     = PROCESSED_DIR / "messages.csv"

# ── Module-level session (same User-Agent as notebook 1) ──────────────────────
SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "BDACA-II-course/1.0 (educational; adashiyj@gmail.com)"
})

print(f"pandas   {pd.__version__}")
print(f"requests {requests.__version__}")
print(f"Session ready — User-Agent: {SESSION.headers['User-Agent']}")

### 2.1 Define helpers

In [ ]:
def read_jsonl(path: Path) -> list[dict]:
    """Read all records from a JSONL file.

    Parameters
    ----------
    path : JSONL file path.

    Returns
    -------
    List of dicts, one per non-empty line. Returns [] if file does not exist.
    """
    if not path.exists():
        return []
    with path.open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def api_get(
    params: dict,
    api: str = API,
    max_retries: int = MAX_RETRIES,
    maxlag: int = MAXLAG,
) -> dict:
    """GET the MediaWiki JSON API via the module-level session.

    Identical to notebook 1 — reproduced here so this notebook is
    self-contained and can be run independently.

    Parameters
    ----------
    params      : MediaWiki API query parameters (without maxlag).
                  A shallow copy is made internally — the caller's dict
                  is never mutated.
    api         : Base URL of the MediaWiki API endpoint.
    max_retries : Maximum number of retry attempts before raising.
    maxlag      : Cooperative DB-lag threshold in seconds.

    Returns
    -------
    dict
        Parsed JSON response from the API.

    Raises
    ------
    RuntimeError
        If the API returns an error body or all retries are exhausted.
    """
    call_params = {**params, "maxlag": str(maxlag)}

    for attempt in range(1, max_retries + 1):
        try:
            response = SESSION.get(
                api,
                params=call_params,
                timeout=REQUEST_TIMEOUT,
            )

            if response.status_code in (429, 503, 504):
                print(
                    f"  HTTP {response.status_code} "
                    f"(attempt {attempt}/{max_retries}), "
                    f"waiting {RETRY_DELAY}s ..."
                )
                time.sleep(RETRY_DELAY)
                continue

            response.raise_for_status()
            result = response.json()

            if "error" in result:
                err = result["error"]
                raise RuntimeError(
                    f"MediaWiki API error [{err.get('code', '?')}]: "
                    f"{err.get('info', err)}"
                )
            if "errors" in result:
                messages = "; ".join(
                    e.get("text", str(e)) for e in result["errors"]
                )
                raise RuntimeError(f"MediaWiki API errors: {messages}")

            return result

        except requests.exceptions.Timeout:
            print(
                f"  Timeout on attempt {attempt}/{max_retries}, "
                f"waiting {RETRY_DELAY}s ..."
            )
            time.sleep(RETRY_DELAY)

        except requests.exceptions.ConnectionError as exc:
            print(
                f"  Connection error (attempt {attempt}/{max_retries}): "
                f"{exc!r}, waiting {RETRY_DELAY}s ..."
            )
            time.sleep(RETRY_DELAY)

    raise RuntimeError(
        f"api_get() failed after {max_retries} retries for params: {params}"
    )

### 2.2 Load and clean the sample

Load `sample_index.json`, convert timestamps to datetime, and produce a clean
newcomer dataframe. No API calls are needed here.

In [ ]:
def load_sample(path: Path = SAMPLE_INDEX_PATH) -> pd.DataFrame:
    """Load the sample index and return a clean newcomer dataframe.

    If the file is not found but a zip archive exists in the same directory,
    it is automatically extracted before loading. This handles the case where
    raw data files were uploaded to the repository as a zip due to size limits.

    Converts first_edit_timestamp to a timezone-aware datetime column and
    computes the retention_check_after column (60 days after first edit),
    which is needed for Step 2.

    Parameters
    ----------
    path : Path to sample_index.json.

    Returns
    -------
    pd.DataFrame with columns: user_id, username, total_edits, session_edits,
    first_edit_timestamp (datetime), retention_check_after (datetime).

    Raises
    ------
    FileNotFoundError
        If sample_index.json is not found and no zip archive exists to
        extract it from.
    """
    # If the file is missing, look for a zip archive in the same directory
    # and extract it automatically — raw data was uploaded zipped due to
    # GitHub's file size limits.
    if not path.exists():
        zip_candidates = list(path.parent.glob("*.zip"))
        if zip_candidates:
            zip_path = zip_candidates[0]
            print(f"  '{path.name}' not found — extracting from {zip_path.name} ...")
            with zipfile.ZipFile(zip_path, "r") as zf:
                zf.extractall(path.parent)
            print(f"  Extraction complete.")
        else:
            raise FileNotFoundError(
                f"Sample index not found at {path} and no zip archive found "
                f"in {path.parent}. Either run notebook 1 to generate the data, "
                f"or download and unzip the data files from the repository."
            )

    with path.open(encoding="utf-8") as f:
        records = json.load(f)

    df = pd.DataFrame(records)

    # Convert timestamp string to timezone-aware datetime for arithmetic.
    df["first_edit_timestamp"] = pd.to_datetime(
        df["first_edit_timestamp"], utc=True
    )

    # Pre-compute the end of the 60-day retention window per newcomer.
    # Retention is checked from this date onwards (see Step 2.3).
    df["retention_check_after"] = (
        df["first_edit_timestamp"] + timedelta(days=RETENTION_DAYS)
    )

    return df


df_newcomers = load_sample()

print(f"Sample size : {len(df_newcomers):,}")
print(f"Columns     : {list(df_newcomers.columns)}")
print(f"Missing values:\n{df_newcomers.isnull().sum()}")
print(f"\nRetention window: {df_newcomers['retention_check_after'].min().date()} "
      f"to {df_newcomers['retention_check_after'].max().date()}")
df_newcomers.head()

### 2.3 Compute retention labels (RQ1 dependent variable)

For each newcomer, check whether they made at least one edit **after** their
60-day window using the `usercontribs` endpoint. This is the binary retention
outcome following Zheng et al. (2019).

The retention window runs from approximately **July 24–31 2025**, reflecting
the narrowed May 25–31 2025 collection window (see note at top of notebook).

In [ ]:
def check_retention(username: str, check_after: datetime) -> int:
    """Check whether a newcomer made at least one edit after the retention window.

    Queries usercontribs with ucstart set to the day after the 60-day window.
    A single edit is sufficient to count as retained, so uclimit=1 minimises
    API cost.

    Parameters
    ----------
    username    : Wikipedia username.
    check_after : Datetime after which edits count as retention.

    Returns
    -------
    1 if the newcomer made at least one edit after check_after, 0 otherwise.
    """
    # Format as ISO 8601 string required by the API.
    check_after_str = check_after.strftime("%Y-%m-%dT%H:%M:%SZ")

    params = {
        "action":        "query",
        "list":          "usercontribs",
        "ucuser":        username,
        "ucstart":       check_after_str,  # only edits after the retention window
        "uclimit":       "1",              # one edit is enough — minimise API cost
        "ucprop":        "ids|timestamp",
        "format":        "json",
        "formatversion": "2",
    }

    data = api_get(params)
    contributions = data.get("query", {}).get("usercontribs", [])

    # 1 = retained (made ≥1 edit after window), 0 = not retained.
    return int(len(contributions) > 0)


def collect_retention_labels(
    df: pd.DataFrame,
    output_path: Path = NEWCOMERS_CSV,
) -> pd.DataFrame:
    """Fetch retention labels for all newcomers and add as a column.

    Skips newcomers already processed if output_path exists (crash-safe).
    Saves progress to disk after every 500 newcomers.

    Parameters
    ----------
    df          : Newcomer dataframe from load_sample().
    output_path : CSV path to save progress to.

    Returns
    -------
    df with a new binary column `retained` (1 = retained, 0 = not retained).
    """
    # Load any previously computed labels to allow resuming after a crash.
    if output_path.exists():
        df_saved = pd.read_csv(output_path)
        already_done = set(df_saved["user_id"])
        print(f"Retention: {len(already_done)} labels already computed.")
    else:
        df_saved = pd.DataFrame()
        already_done = set()

    results = []
    to_process = df[~df["user_id"].isin(already_done)]
    print(f"Retention: {len(to_process)} newcomers to check.")

    for i, row in enumerate(to_process.itertuples(index=False)):
        retained = check_retention(row.username, row.retention_check_after)
        results.append({"user_id": row.user_id, "retained": retained})

        # Save progress every 500 newcomers so a crash loses little work.
        if (i + 1) % 500 == 0:
            df_partial = pd.DataFrame(results)
            df_combined = pd.concat([df_saved, df_partial], ignore_index=True)
            df_combined.to_csv(output_path, index=False)
            print(f"  Progress: {i + 1}/{len(to_process)} checked, saved.")

        time.sleep(INTER_REQUEST_DELAY)

    # Final save.
    df_new_labels = pd.DataFrame(results)
    df_all_labels = pd.concat([df_saved, df_new_labels], ignore_index=True)
    df_all_labels.to_csv(output_path, index=False)

    # Merge retention labels back onto the newcomer dataframe.
    df_with_retention = df.merge(df_all_labels[["user_id", "retained"]], on="user_id", how="left")

    retained_count = df_with_retention["retained"].sum()
    retention_rate = retained_count / len(df_with_retention)
    print(f"\nRetention labels complete.")
    print(f"  Retained    : {int(retained_count):,}  ({retention_rate:.1%})")
    print(f"  Not retained: {int(len(df_with_retention) - retained_count):,}")

    return df_with_retention


df_newcomers = collect_retention_labels(df_newcomers)
df_newcomers[["username", "session_edits", "retained"]].head()

### 2.4 Load bot role classification and match senders

Load Zheng et al.'s (2019) bot classification file and match each talk page
message sender against it. Unmatched senders are cross-checked against
Wikipedia's live bot category before being classified as human or unidentified.

In [ ]:
def load_bot_classification(path: Path = BOT_CLASSIFICATION_PATH) -> dict[str, str]:
    """Load Zheng et al. (2019) bot role classification file.

    Returns a dict mapping bot username -> role label for fast lookup.

    Parameters
    ----------
    path : Path to the .xlsx classification file. Defaults to
           BOT_CLASSIFICATION_PATH defined in the configuration cell.

    Returns
    -------
    Dict of {username: role_label} for all 1,601 classified bots.

    Raises
    ------
    FileNotFoundError
        If the classification file is not found at the expected path.
        The file is included in the repository under data/raw/ — ensure
        the working directory is set correctly in the configuration cell.
    """
    if not path.exists():
        raise FileNotFoundError(
            f"Bot classification file not found at {path}.\n"
            "This file is included in the repository under data/raw/.\n"
            "Check that the working directory is set correctly in cell 2.0, "
            "or verify the file was not accidentally excluded from the repo."
        )

    df_bots = pd.read_excel(path)

    # Normalise column names to lowercase to guard against inconsistent
    # capitalisation across Excel versions.
    df_bots.columns = df_bots.columns.str.lower().str.strip()

    # Build username -> role lookup. Strip whitespace to avoid silent
    # mismatches caused by trailing spaces in the Excel file.
    bot_role_lookup = {
        str(row["bot_name"]).strip(): str(row["role"]).strip()
        for _, row in df_bots.iterrows()
        if pd.notna(row.get("bot_name")) and pd.notna(row.get("role"))
    }

    print(f"Bot classification file loaded: {len(bot_role_lookup):,} bots.")
    role_counts = pd.Series(bot_role_lookup.values()).value_counts()
    print(f"Role distribution:\n{role_counts.to_string()}")

    return bot_role_lookup

In [ ]:
def fetch_live_bot_usernames() -> set[str]:
    """Fetch all usernames currently listed in Wikipedia's bot category.

    Used to cross-check unmatched senders — if a sender appears here but
    not in Zheng et al. (2019), they are classified as an unidentified bot
    (created after February 2019) rather than a human editor.

    Returns
    -------
    Set of bot usernames from the live Wikipedia bot category.
    """
    bot_usernames = set()

    params = {
        "action":      "query",
        "list":        "categorymembers",
        "cmtitle":     WIKI_BOT_CATEGORY,
        "cmtype":      "page",
        "cmlimit":     "500",
        "cmnamespace": "2",   # User namespace only
        "format":      "json",
        "formatversion": "2",
    }

    while True:
        data    = api_get(params)
        members = data.get("query", {}).get("categorymembers", [])

        for member in members:
            # Strip "User:" prefix to get the bare username.
            title = member.get("title", "")
            if title.startswith("User:"):
                bot_usernames.add(title[len("User:"):])

        if "continue" not in data:
            break
        params.update(data["continue"])
        time.sleep(INTER_PAGE_DELAY)

    print(f"Live bot category: {len(bot_usernames):,} bots fetched.")
    return bot_usernames


live_bot_usernames = fetch_live_bot_usernames()

In [ ]:
def classify_sender(
    sender: str,
    bot_role_lookup: dict[str, str],
    live_bot_usernames: set[str],
) -> str:
    """Classify a message sender as a bot role, human, or unidentified bot.

    Classification priority:
      1. If sender matches Zheng et al. (2019) lookup → return role label.
      2. If sender appears in live Wikipedia bot category → 'unidentified_bot'.
      3. Otherwise → 'human'.

    Parameters
    ----------
    sender             : Wikipedia username of the message sender.
    bot_role_lookup    : Dict of {username: role} from Zheng et al. (2019).
    live_bot_usernames : Set of usernames from the live bot category.

    Returns
    -------
    Role label string: one of the nine Zheng et al. roles, 'human', or
    'unidentified_bot'.
    """
    if not sender:
        return "human"  # missing sender treated as human by default

    sender_clean = sender.strip()

    if sender_clean in bot_role_lookup:
        return bot_role_lookup[sender_clean]

    # Unmatched sender — check against live bot category before classifying
    # as human. This catches bots created after February 2019 that are absent
    # from Zheng et al.'s (2019) taxonomy (see dossier, bias section).
    if sender_clean in live_bot_usernames:
        return "unidentified_bot"

    return "human"

In [ ]:
# Quick test: confirm classify_sender works with known examples.
test_senders = ["ClueBot NG", "HasteurBot", "SomeRandomHuman123"]
for sender in test_senders:
    label = classify_sender(sender, bot_role_lookup, live_bot_usernames)
    print(f"  {sender:30s} → {label}")

### 2.5 Build the message-level dataframe (RQ2 unit of analysis)

Flatten talk page messages into one row per message, classify each sender,
and save as `messages.csv`. This is the input for ConvoKit politeness scoring
in the next step.

In [ ]:
def build_message_dataframe(
    talkpage_path: Path,
    bot_role_lookup: dict[str, str],
    live_bot_usernames: set[str],
) -> pd.DataFrame:
    """Flatten talkpage_messages.jsonl into a message-level dataframe.

    Each row represents one message received by a newcomer. The sender is
    classified using classify_sender().

    Parameters
    ----------
    talkpage_path      : Path to talkpage_messages.jsonl.
    bot_role_lookup    : Zheng et al. (2019) bot role lookup dict.
    live_bot_usernames : Set of live Wikipedia bot usernames.

    Returns
    -------
    pd.DataFrame with columns: user_id, username, revid, timestamp,
    sender, sender_type, sizediff.
    """
    talkpage_records = read_jsonl(talkpage_path)
    rows = []

    for record in talkpage_records:
        user_id  = record["user_id"]
        username = record["username"]

        for msg in record.get("messages", []):
            sender      = msg.get("sender", "")
            sender_type = classify_sender(
                sender, bot_role_lookup, live_bot_usernames
            )
            rows.append({
                "user_id":     user_id,
                "username":    username,
                "revid":       msg.get("revid"),
                "timestamp":   msg.get("timestamp", ""),
                "sender":      sender,
                "sender_type": sender_type,
                "sizediff":    msg.get("sizediff", 0),
            })

    df_messages = pd.DataFrame(rows)

    print(f"Total messages        : {len(df_messages):,}")
    print(f"\nSender type distribution:")
    print(df_messages["sender_type"].value_counts().to_string())

    # Report unidentified bots separately so the proportion can be assessed
    # as a limitation (following Sen et al., 2021).
    n_unidentified = (df_messages["sender_type"] == "unidentified_bot").sum()
    pct_unidentified = n_unidentified / max(len(df_messages), 1)
    print(f"\nUnidentified bots: {n_unidentified:,} messages ({pct_unidentified:.1%})")
    print("(These are excluded from role-based analyses — see dossier bias section.)")

    return df_messages


df_messages = build_message_dataframe(
    TALKPAGE_JSONL, bot_role_lookup, live_bot_usernames
)
df_messages.to_csv(MESSAGES_CSV, index=False)
print(f"\nSaved to {MESSAGES_CSV}")
df_messages.head()

### 2.6 Build the newcomer-level analysis dataframe (RQ1)

Aggregate the message dataframe to the newcomer level. For each newcomer,
create one binary contact dummy per sender type (1 = received ≥1 message
from that type, 0 = did not). This is the analysis dataset for the RQ1
logistic regression.

In [ ]:
def build_analysis_dataset(
    df_newcomers: pd.DataFrame,
    df_messages: pd.DataFrame,
) -> pd.DataFrame:
    """Aggregate messages to newcomer level and create contact dummies.

    For each sender type (bot role or human), a binary contact dummy is
    created indicating whether the newcomer received at least one message
    from that sender type within the 60-day window. Unidentified bots are
    excluded from role-based dummies and reported as a separate column.

    Parameters
    ----------
    df_newcomers : Newcomer dataframe with retention labels (from Step 2.3).
    df_messages  : Message-level dataframe (from Step 2.5).

    Returns
    -------
    pd.DataFrame with one row per newcomer, containing retention label,
    session_edits control variable, and one binary contact dummy per
    sender type.
    """
    # Identify all sender types except unidentified bots — those are excluded
    # from role-based analysis following the dossier bias mitigation strategy.
    role_types = [
        t for t in df_messages["sender_type"].unique()
        if t != "unidentified_bot"
    ]

    # Build one binary contact dummy per sender type at the newcomer level.
    contact_dummies = {}
    for sender_type in role_types:
        # Clean column name: spaces and slashes cause issues in regression.
        col_name = f"contact_{sender_type.lower().replace(' ', '_')}"
        contacted_users = set(
            df_messages.loc[
                df_messages["sender_type"] == sender_type, "user_id"
            ]
        )
        contact_dummies[col_name] = df_newcomers["user_id"].isin(contacted_users).astype(int)

    # Unidentified bot contact is recorded separately for transparency.
    unidentified_users = set(
        df_messages.loc[
            df_messages["sender_type"] == "unidentified_bot", "user_id"
        ]
    )
    contact_dummies["contact_unidentified_bot"] = (
        df_newcomers["user_id"].isin(unidentified_users).astype(int)
    )

    df_dummies = pd.DataFrame(contact_dummies)
    df_analysis = pd.concat(
        [df_newcomers[["user_id", "username", "session_edits", "retained"]].reset_index(drop=True),
         df_dummies.reset_index(drop=True)],
        axis=1,
    )

    print(f"Analysis dataset: {len(df_analysis):,} rows x {len(df_analysis.columns)} columns")
    print(f"\nContact dummy columns: {[c for c in df_analysis.columns if c.startswith('contact_')]}")
    print(f"\nRetention rate: {df_analysis['retained'].mean():.1%}")
    print(f"\nContact rates:")
    contact_cols = [c for c in df_analysis.columns if c.startswith("contact_")]
    print(df_analysis[contact_cols].mean().sort_values(ascending=False).to_string())

    return df_analysis


df_analysis = build_analysis_dataset(df_newcomers, df_messages)
df_analysis.to_csv(ANALYSIS_CSV, index=False)
print(f"\nSaved to {ANALYSIS_CSV}")
df_analysis.head()

### 2.7 Summary
Quick sanity check of all output files before proceeding to analysis.

In [ ]:
print("=" * 55)
print("Preprocessing complete — output files")
print("=" * 55)

for path in (NEWCOMERS_CSV, MESSAGES_CSV, ANALYSIS_CSV):
    if path.exists():
        df_check = pd.read_csv(path)
        print(f"  {path.name:35s} {len(df_check):>6,} rows x {len(df_check.columns)} cols")
    else:
        print(f"  {path.name:35s} NOT FOUND")

print()
print("Next step: run 03_politeness_scoring.ipynb to extract")
print("ConvoKit politeness features from messages.csv.")